In [ ]:
!pip install -q --upgrade plotly ipywidgets
from google.colab import output
output.enable_custom_widget_manager()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

def update_plot(epsilon, interval_range, max_n):
    # Calculations
    n_values = np.arange(1, max_n + 1)

    # Hoeffding's bound formula: 2 * exp(-2 * n * eps^2 / (b - a)^2)
    hoeffding_bound = 2 * np.exp((-2 * n_values * (epsilon ** 2)) / (interval_range ** 2))

    # Probability bound cannot logically exceed 1
    hoeffding_bound_clipped = np.clip(hoeffding_bound, 0, 1)

    # Find sample size where error probability drops below 5%
    target_prob = 0.05
    n_5_percent = np.argmax(hoeffding_bound <= target_prob) + 1

    # Create interactive plot with Plotly
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=n_values,
        y=hoeffding_bound_clipped,
        mode='lines',
        name="Hoeffding Bound",
        line=dict(color='#E63946', width=3),
        hovertemplate="Sample Size (n): %{x}<br>Max Error Prob: %{y:.4f}<extra></extra>"
    ))

    # Reference line for 5% error probability threshold
    fig.add_hline(
        y=0.05,
        line_dash="dash",
        line_color="gray",
        annotation_text="5% Probability Threshold",
        annotation_position="top right"
    )

    title_text = f"Hoeffding Bound: Upper Limit on P(|X̄ - μ| ≥ {epsilon:.2f})"
    if hoeffding_bound[n_5_percent - 1] <= target_prob:
        title_text += f"<br><sup>Sample size needed for ≤ 5% error bound: <b>n = {n_5_percent:,}</b></sup>"

    fig.update_layout(
        title=title_text,
        xaxis_title="Sample Size (n)",
        yaxis_title="Upper Bound on Error Probability",
        yaxis=dict(range=[0, 1.05]),
        template="plotly_white",
        hovermode="x unified",
        height=500
    )

    fig.show()

# Set up Colab interactive sliders
epsilon_slider = widgets.FloatSlider(
    value=0.10, min=0.01, max=0.50, step=0.01,
    description='Tolerance (ε):', style={'description_width': 'initial'}
)

range_slider = widgets.FloatSlider(
    value=1.0, min=0.5, max=5.0, step=0.1,
    description='Bound (b - a):', style={'description_width': 'initial'}
)

max_n_slider = widgets.IntSlider(
    value=1000, min=100, max=5000, step=100,
    description='Max Sample Size (n):', style={'description_width': 'initial'}
)

# Connect controls to update function
out = widgets.interactive_output(update_plot, {
    'epsilon': epsilon_slider,
    'interval_range': range_slider,
    'max_n': max_n_slider
})

# Display UI in Colab
ui = widgets.VBox([epsilon_slider, range_slider, max_n_slider])

display(ui, out)


Output()

In [ ]:
#update_plot(epsilon_slider.value, range_slider.value, max_n_slider.value)